# LiDAR PCAP Frame Clustering

Upload a classic Ethernet `.pcap` containing Hesai protocol v1.4 LiDAR packets. The decoder uses the JT128 128-channel elevation and azimuth calibration, validates packet sizes, and separates frames at a full azimuth rotation. The default run is limited to the first **3 frames**. Distance filtering and seeded point subsampling run before clustering to keep processing manageable.

Per-frame point arrays and labels are retained in `app_state['results']` so a future tracker can associate cluster centroids across frames.

Dependencies: `dpkt`, `numpy`, `matplotlib`, `ipywidgets`, `scikit-learn`, and `hdbscan`. If imports fail, install them in the active kernel with `%pip install dpkt numpy matplotlib ipywidgets scikit-learn hdbscan`, then restart the kernel.

In [1]:
import io
import math
import struct

import dpkt
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering

## Hesai PCAP decoding

In [2]:
import csv
import io
import struct
from pathlib import Path

def load_jt128_calibration(path):
    calibration = {}
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as calibration_file:
        for row in csv.DictReader(calibration_file):
            try:
                channel = int(row["Channel"])
                elevation = float(row["Elevation"])
                azimuth = float(row["Azimuth"])
            except (KeyError, TypeError, ValueError):
                continue
            calibration[channel] = (elevation, azimuth)

    if sorted(calibration) != list(range(1, 129)):
        raise ValueError(f"Expected 128 JT128 calibration rows in {path}.")

    elevations = np.array([calibration[channel][0] for channel in range(1, 129)], dtype=float)
    azimuth_offsets = np.array([calibration[channel][1] for channel in range(1, 129)], dtype=float)
    return elevations, azimuth_offsets


CALIBRATION_PATH = Path("../Lidar setup/JT128_angle_correction.csv")
if not CALIBRATION_PATH.exists():
    CALIBRATION_PATH = Path("Lidar setup/JT128_angle_correction.csv")
VERTICAL_ANGLES_DEG, AZIMUTH_OFFSETS_DEG = load_jt128_calibration(CALIBRATION_PATH)


def decode_lidar_packet(payload):
    if len(payload) < 12 or payload[:4] != b"\xee\xff\x01\x04":
        return []

    laser_count, block_count, _, distance_unit_mm, _, status_flags = payload[6:12]
    if laser_count != len(VERTICAL_ANGLES_DEG):
        raise ValueError(f"Packet has {laser_count} channels, but the calibration has {len(VERTICAL_ANGLES_DEG)}.")

    unit_size = 3 + bool(status_flags & 0x10) + bool(status_flags & 0x20) + bool(status_flags & 0x40)
    block_size = 2 + laser_count * unit_size
    optional_size = 4
    optional_size += 17 if status_flags & 0x04 else 0
    optional_size += 26
    optional_size += 4 if status_flags & 0x01 else 0
    optional_size += 22 if status_flags & 0x02 else 0
    optional_size += 4
    optional_size += 32 if status_flags & 0x08 else 0
    expected_size = 12 + block_count * block_size + optional_size
    if len(payload) != expected_size:
        raise ValueError(f"Unexpected Hesai v1.4 payload size {len(payload)}; expected {expected_size}.")

    distance_unit_m = distance_unit_mm / 1000.0
    vertical_angles = np.radians(VERTICAL_ANGLES_DEG)
    decoded_blocks = []
    block_offset = 12
    for _ in range(block_count):
        azimuth_deg = int.from_bytes(payload[block_offset:block_offset + 2], "little") / 100.0
        block_offset += 2
        azimuths = np.radians((azimuth_deg + AZIMUTH_OFFSETS_DEG) % 360.0)
        distances = np.empty(laser_count, dtype=float)

        for channel in range(laser_count):
            raw_distance = struct.unpack_from("<H", payload, block_offset)[0]
            distances[channel] = raw_distance * distance_unit_m
            block_offset += unit_size

        horizontal = distances * np.cos(vertical_angles)
        points = np.column_stack((
            horizontal * np.sin(azimuths),
            horizontal * np.cos(azimuths),
            distances * np.sin(vertical_angles),
        ))
        decoded_blocks.append((azimuth_deg, points[distances > 0]))

    return decoded_blocks


def pcap_to_frames(file_bytes, max_frames=3):
    try:
        reader = dpkt.pcap.Reader(io.BytesIO(file_bytes))
    except (ValueError, dpkt.dpkt.NeedData, struct.error) as error:
        raise ValueError("Unsupported capture format. Upload a classic .pcap file, not .pcapng.") from error
    if reader.datalink() != dpkt.pcap.DLT_EN10MB:
        raise ValueError("Unsupported PCAP link type. Expected Ethernet frames carrying IP/UDP LiDAR packets.")
    if max_frames < 1:
        raise ValueError("Frame count must be at least 1.")

    frames = []
    frame_blocks = []
    previous_azimuth = None
    udp_packet_count = 0
    hesai_packet_count = 0
    completed_frame_limit = False

    for _, raw_packet in reader:
        try:
            ethernet = dpkt.ethernet.Ethernet(raw_packet)
            network = ethernet.data
            if isinstance(network, dpkt.ip.IP):
                transport = network.data
            elif isinstance(network, dpkt.ip6.IP6):
                transport = network.data
            else:
                continue
            if not isinstance(transport, dpkt.udp.UDP):
                continue
            payload = bytes(transport.data)
        except (AttributeError, dpkt.dpkt.NeedData, ValueError):
            continue

        udp_packet_count += 1
        if payload[:4] != b"\xee\xff\x01\x04":
            continue
        hesai_packet_count += 1
        try:
            decoded_blocks = decode_lidar_packet(payload)
        except ValueError:
            continue

        for azimuth, points in decoded_blocks:
            if previous_azimuth is not None and previous_azimuth - azimuth > 180:
                if frame_blocks:
                    frames.append(np.vstack(frame_blocks))
                    frame_blocks = []
                if len(frames) >= max_frames:
                    completed_frame_limit = True
                    break
            if len(points):
                frame_blocks.append(points)
            previous_azimuth = azimuth

        if completed_frame_limit:
            break

    if frame_blocks and len(frames) < max_frames:
        frames.append(np.vstack(frame_blocks))
    if not frames:
        if not udp_packet_count:
            raise ValueError("No Ethernet/IP/UDP packets found. Expected a classic Ethernet PCAP containing UDP LiDAR data.")
        if not hesai_packet_count:
            raise ValueError("No Hesai protocol v1.4 packets found. Expected UDP payloads beginning EE FF 01 04.")
        raise ValueError("Hesai packets were found, but none had a valid v1.4 layout and 128-channel JT128 data.")
    return frames

## Filtering, subsampling, and clustering

In [3]:
def prepare_points(points, min_distance, max_distance, keep_fraction, seed):
    distances = np.linalg.norm(points, axis=1)
    filtered = points[(distances >= min_distance) & (distances <= max_distance)]
    if len(filtered) and keep_fraction < 1.0:
        keep_count = max(1, int(np.ceil(len(filtered) * keep_fraction)))
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(len(filtered), size=keep_count, replace=False))
        filtered = filtered[indices]
    return filtered


def cluster_points(points, method, cluster_count, dbscan_eps, min_samples, hdbscan_min_cluster_size, spectral_neighbors):
    point_count = len(points)
    if point_count == 0:
        return np.empty(0, dtype=int)
    if method == 'K-means':
        return KMeans(n_clusters=min(cluster_count, point_count), n_init=10, random_state=42).fit_predict(points)
    if method == 'Spectral':
        if point_count < 2:
            return np.zeros(point_count, dtype=int)
        count = min(cluster_count, point_count - 1)
        neighbors = min(spectral_neighbors, point_count - 1)
        return SpectralClustering(n_clusters=count, affinity='nearest_neighbors', n_neighbors=neighbors, assign_labels='kmeans', random_state=42).fit_predict(points)
    if method == 'DBSCAN':
        return DBSCAN(eps=dbscan_eps, min_samples=min_samples).fit_predict(points)
    if method == 'HDBSCAN':
        if point_count < 2:
            return np.full(point_count, -1, dtype=int)
        return hdbscan.HDBSCAN(min_cluster_size=min(hdbscan_min_cluster_size, point_count), min_samples=min_samples).fit_predict(points)
    raise ValueError(f'Unknown clustering method: {method}')


def cluster_frame(points, method, settings, seed):
    selected = prepare_points(points, settings['min_distance'], settings['max_distance'], settings['keep_fraction'], seed)
    labels = cluster_points(selected, method, settings['cluster_count'], settings['dbscan_eps'], settings['min_samples'], settings['hdbscan_min_cluster_size'], settings['spectral_neighbors'])
    cluster_labels = set(labels.tolist()) - {-1}
    return {
        'points': selected,
        'labels': labels,
        'decoded_count': len(points),
        'used_count': len(selected),
        'cluster_count': len(cluster_labels),
        'noise_count': int(np.count_nonzero(labels == -1)),
    }

## Interactive app

In [4]:
app_state = {'frames': [], 'results': []}
upload = widgets.FileUpload(accept='.pcap', multiple=False, description='Choose PCAP')
frame_count = widgets.IntSlider(value=3, min=1, max=200, step=1, description='Frames', continuous_update=False)
method = widgets.Dropdown(options=['K-means', 'Spectral', 'DBSCAN', 'HDBSCAN'], value='DBSCAN', description='Algorithm')
min_distance = widgets.FloatSlider(value=0.0, min=0.0, max=150.0, step=0.5, description='Min range (m)', continuous_update=False)
max_distance = widgets.FloatSlider(value=80.0, min=1.0, max=200.0, step=0.5, description='Max range (m)', continuous_update=False)
keep_fraction = widgets.FloatSlider(value=0.10, min=0.01, max=1.0, step=0.01, description='Keep points', readout_format='.0%', continuous_update=False)
cluster_count = widgets.IntSlider(value=4, min=2, max=30, step=1, description='K clusters', continuous_update=False)
dbscan_eps = widgets.FloatSlider(value=0.5, min=0.05, max=5.0, step=0.05, description='DBSCAN eps', continuous_update=False)
min_samples = widgets.IntSlider(value=5, min=2, max=50, step=1, description='Min samples', continuous_update=False)
hdbscan_min_cluster_size = widgets.IntSlider(value=10, min=2, max=100, step=1, description='HDB min size', continuous_update=False)
spectral_neighbors = widgets.IntSlider(value=10, min=2, max=50, step=1, description='Spectral neighbors', continuous_update=False)
frame_picker = widgets.Dropdown(options=[], description='Preview frame')
run_button = widgets.Button(description='Decode and cluster', button_style='primary', icon='play')
status = widgets.HTML(value='Choose a Hesai v1.4 PCAP. The initial run processes at most 3 rotations.')
summary_output = widgets.Output()
plot_output = widgets.Output()

def get_uploaded_file():
    if not upload.value:
        raise ValueError('Choose a .pcap file first.')
    uploaded = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    return uploaded['name'], uploaded['content']


def render_summary():
    rows = ''.join(
        f"<tr><td>{index}</td><td>{result['decoded_count']:,}</td><td>{result['used_count']:,}</td><td>{result['cluster_count']}</td><td>{result['noise_count']:,}</td></tr>"
        for index, result in enumerate(app_state['results'])
    )
    table = (
        "<table style='border-collapse:collapse'>"
        "<tr><th>Frame</th><th>Decoded</th><th>Clustered</th><th>Clusters</th><th>Noise</th></tr>"
        + rows + "</table>"
    )
    with summary_output:
        clear_output(wait=True)
        display(widgets.HTML(table))


def render_selected_frame(change=None):
    if not app_state['results'] or frame_picker.value is None:
        return
    frame_index = int(frame_picker.value)
    result = app_state['results'][frame_index]
    points, labels = result['points'], result['labels']
    with plot_output:
        clear_output(wait=True)
        if not len(points):
            display(widgets.HTML('No points remain after distance filtering.'))
            return
        figure = plt.figure(figsize=(8, 6))
        axis = figure.add_subplot(111, projection='3d')
        noise = labels == -1
        if np.any(noise):
            axis.scatter(points[noise, 0], points[noise, 1], points[noise, 2], c='#777777', s=3, label='Noise')
        clustered = ~noise
        if np.any(clustered):
            axis.scatter(points[clustered, 0], points[clustered, 1], points[clustered, 2], c=labels[clustered], cmap='tab20', s=4)
        axis.set(title=f'{method.value} - frame {frame_index}', xlabel='X (m)', ylabel='Y (m)', zlabel='Z (m)')
        plt.tight_layout()
        plt.show()


def run_app(_):
    try:
        filename, file_bytes = get_uploaded_file()
        if min_distance.value > max_distance.value:
            raise ValueError('Minimum range must not exceed maximum range.')
        status.value = '<span style="color:#555">Decoding rotations and clustering...</span>'
        app_state['frames'] = pcap_to_frames(file_bytes, max_frames=frame_count.value)
        settings = {
            'min_distance': min_distance.value,
            'max_distance': max_distance.value,
            'keep_fraction': keep_fraction.value,
            'cluster_count': cluster_count.value,
            'dbscan_eps': dbscan_eps.value,
            'min_samples': min_samples.value,
            'hdbscan_min_cluster_size': hdbscan_min_cluster_size.value,
            'spectral_neighbors': spectral_neighbors.value,
        }
        app_state['results'] = [
            cluster_frame(points, method.value, settings, seed=42 + index)
            for index, points in enumerate(app_state['frames'])
        ]
        frame_picker.options = [(f'Frame {index}', index) for index in range(len(app_state['results']))]
        frame_picker.value = 0
        render_summary()
        render_selected_frame()
        status.value = f"<span style='color:#176b3a'>{filename}: clustered {len(app_state['results'])} frame(s) with {method.value}.</span>"
    except Exception as error:
        app_state['frames'] = []
        app_state['results'] = []
        status.value = f'<span style="color:#a11">{type(error).__name__}: {error}</span>'
        with summary_output:
            clear_output(wait=True)
        with plot_output:
            clear_output(wait=True)


def on_method_change(change):
    parameter_controls.layout.display = 'flex' if change['new'] in ('K-means', 'Spectral') else 'none'
    spectral_controls.layout.display = 'flex' if change['new'] == 'Spectral' else 'none'
    dbscan_controls.layout.display = 'flex' if change['new'] == 'DBSCAN' else 'none'
    hdbscan_controls.layout.display = 'flex' if change['new'] == 'HDBSCAN' else 'none'
    render_selected_frame()


parameter_controls = widgets.HBox([cluster_count])
spectral_controls = widgets.HBox([spectral_neighbors])
dbscan_controls = widgets.HBox([dbscan_eps, min_samples])
hdbscan_controls = widgets.HBox([hdbscan_min_cluster_size, min_samples])
method.observe(on_method_change, names='value')
frame_picker.observe(render_selected_frame, names='value')
run_button.on_click(run_app)
on_method_change({'new': method.value})
display(widgets.VBox([
    upload,
    widgets.HBox([frame_count]),
    widgets.HBox([method, keep_fraction]),
    widgets.HBox([min_distance, max_distance]),
    parameter_controls, spectral_controls, dbscan_controls, hdbscan_controls,
    run_button, status, summary_output, frame_picker, plot_output,
]))
print('LiDAR PCAP clustering app ready. Initial frame limit: 3.')

LiDAR PCAP clustering app ready. Initial frame limit: 3.
